# Домашнее задание 4 — Тематическое моделирование (BERTopic)
**Датасет:** Lenta.Ru News  
**Задача:** классификация текстов по топикам с помощью BERTopic

In [ ]:
%%capture
!pip install corus pymorphy3 nltk bertopic gensim sentence-transformers

## 1. Загрузка данных
Загружаем набор данных lenta-ru-news с помощью библиотеки Corus.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

!wget -q https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

from corus import load_lenta

path = 'lenta-ru-news.csv.gz'
records = load_lenta(path)

data = []
for record in records:
    data.append({
        'title': record.title,
        'text': record.text,
        'topic': record.topic
    })

df = pd.DataFrame(data)
print(f"Всего: {len(df)}")
df.head()

Всего: 739351


,title,text,topic
0,Названы регионы России с самой высокой смертно...,Вице-премьер по социальным вопросам Татьяна Го...,Россия
1,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт
2,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия
3,В США раскрыли сумму расходов на расследование...,С начала расследования российского вмешательст...,Мир
4,Хакеры рассказали о планах Великобритании зами...,Хакерская группировка Anonymous опубликовала н...,Мир


In [ ]:
# Семплируем 100к записей, из них возьмём 20к для обработки (тк лемматизация довольно тяжёлая операция)
df_sampled = df.sample(n=30000, random_state=42).reset_index(drop=True)
# df_sampled = df_sampled.head(20000).copy()
print(f"Размер выборки для моделирования: {len(df_sampled)}")
print(f"Распределение топиков:\n{df_sampled['topic'].value_counts().head(10)}")

Размер выборки для моделирования: 30000
Распределение топиков:
topic
Россия             6532
Мир                5505
Экономика          3262
Спорт              2589
Наука и техника    2204
Культура           2183
Бывший СССР        2086
Интернет и СМИ     1889
Из жизни           1109
Дом                 841
Name: count, dtype: int64


In [ ]:
df_sampled

## 2. Предобработка текстов (2 балла)

**Необходимость предобработки для тематического моделирования:**

Данные требуют значительной предобработки по нескольким причинам:
- **Морфологическое разнообразие русского языка:** одно слово может иметь десятки словоформ, что размывает частотные распределения. Лемматизация (pymorphy3) сводит словоформы к начальной форме.
- **Шум:** HTML-теги, знаки препинания, цифры не несут семантической нагрузки для тематического моделирования.
- **Стоп-слова:** предлоги, союзы, местоимения и т.д. встречаются во всех темах одинаково и мешают выделению различий.

Пайплайн предобработки:
1. Удаление HTML-тегов
2. Удаление пунктуации и цифр
3. Приведение к нижнему регистру
4. Удаление стоп-слов (NLTK)
5. Лемматизация (pymorphy3)
6. Фильтрация коротких токенов (длина ≤ 2)

In [4]:
import re
import string
import nltk
from nltk.corpus import stopwords
from pymorphy3 import MorphAnalyzer

nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('russian'))
morph = MorphAnalyzer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<.*?>', '', text)# HTML
    text = re.sub(f"[{re.escape(string.punctuation)}0-9]", ' ', text)# пунктуация+цифры
    words = text.lower().split()
    cleaned_tokens = []
    for word in words:
        if word not in stop_words and len(word) > 2:
            parsed = morph.parse(word)
            if parsed:
                cleaned_tokens.append(parsed[0].normal_form)
    return " ".join(cleaned_tokens)

print(f"Предобработка {len(df_sampled)} текстов...")
df_sampled['cleaned_text'] = df_sampled['text'].apply(preprocess_text)
df_sampled[['text', 'cleaned_text']].head(3)

Предобработка 30000 текстов...


,text,cleaned_text
0,Египетский перевозчик EgyptAir сообщил о возмо...,египетский перевозчик egyptair сообщить возмож...
1,Глава Красногорского района Московской области...,глава красногорский район московский область б...
2,Депутат Виталий Милонов внес в Госдуму законоп...,депутат виталий милон внести госдума законопро...


In [5]:
df_sampled

,title,text,topic,cleaned_text
0,EgyptAir объявила о подорожании билетов,Египетский перевозчик EgyptAir сообщил о возмо...,Путешествия,египетский перевозчик egyptair сообщить возмож...
1,Глава Красногорского района Подмосковья ушел в...,Глава Красногорского района Московской области...,Россия,глава красногорский район московский область б...
2,Милонов предложил запретить россиянам сидеть в...,Депутат Виталий Милонов внес в Госдуму законоп...,Россия,депутат виталий милон внести госдума законопро...
3,Женщинам в детородном возрасте разрешили посещ...,Верховный суд Индии разрешил женщинам в фертил...,Мир,верховный суд индия разрешить женщина фертильн...
4,Россиянам пообещали дешевый хлеб,Россиянам не стоит бояться роста цен на хлеб —...,Экономика,россиянин стоить бояться рост цена хлеб никако...
...,...,...,...,...
29995,Британские власти запретили две исламистские г...,"В Великобритании впервые был применен закон, з...",Россия,великобритания впервые применить закон запреща...
29996,Российский спортсмен побил мировой рекорд по п...,Российский спортсмен Виктор Филиппов побил мир...,Спорт,российский спортсмен виктор филипп побить миро...
29997,В московской двенадцатиэтажке взорвался газ,В двенадцатиэтажном жилом доме на северо-восто...,Россия,двенадцатиэтажный жилой дом северо восток моск...
29998,В Египте перед храмом в Луксоре подорвался сме...,В Египте смертник подорвался у Карнакского хра...,Мир,египет смертник подорваться карнакский храм лу...


## 3. BERTopic: выбор компонентов пайплайна (3 балла)

### Обоснование выбора каждого элемента:

| Компонент | Выбор | Обоснование |
|-----------|-------|-------------|
| **Энкодер** | `paraphrase-multilingual-MiniLM-L12-v2` | Мультиязычная модель, обученная на параллельных корпусах; хорошо поддерживает русский язык. Компактна (120M параметров) и быстра при хорошем качестве эмбеддингов. |
| **Снижение размерности** | UMAP | Сохраняет как локальную, так и глобальную структуру данных, в отличие от t-SNE (только локальная) или PCA (линейное). Стандарт для BERTopic. |
| **Кластеризация** | HDBSCAN | Плотностная кластеризация: не требует заранее задавать число кластеров, выделяет кластеры разного размера и помечает шум (Topic -1). Подходит лучше, чем K-Means (фиксированное k) или DBSCAN (одна плотность). |
| **Токенизация** | CountVectorizer + русские стоп-слова | c-TF-IDF в BERTopic работает поверх CountVectorizer. Дополнительная фильтрация стоп-слов на этапе c-TF-IDF убирает остаточный шум. |
| **Постобработка** | Встроенный c-TF-IDF BERTopic | Автоматически ранжирует токены внутри каждого топика по релевантности. |

## 4. Настройка гиперпараметров (1 балл)

### Обоснование гиперпараметров:

- **UMAP `n_neighbors=15`** — баланс между локальной и глобальной структурой. Слишком маленькое значение (5) создаёт фрагментированные кластеры, слишком большое (50) — размывает границы тем.
- **UMAP `n_components=5`** — 5 измерений достаточно для передачи информации в HDBSCAN, при этом снижается проклятие размерности (384 → 5).
- **UMAP `min_dist=0.0`** — позволяет точкам сжиматься максимально плотно, что улучшает кластеризацию.
- **HDBSCAN `min_cluster_size=50`** — минимальный размер кластера. Для 20k документов значение 50 обеспечивает стабильные, не слишком мелкие темы.
- **HDBSCAN `cluster_selection_method='eom'`** — Excess of Mass: выбирает кластеры, максимизируя стабильность, что даёт более интерпретируемые темы.
- **`calculate_probabilities=True`** — необходимо для визуализации распределения тем по документам.

In [6]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

# Энкодер
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Снижение размерности
umap_model = UMAP(
    n_neighbors=15, n_components=5,
    min_dist=0.0, metric='cosine', random_state=42
)

# Кластеризация
hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric='euclidean',
    cluster_selection_method='eom', prediction_data=True
)

# Токенизация
vectorizer_model = CountVectorizer(stop_words=list(stop_words))

print("Компоненты пайплайна инициализированы.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Компоненты пайплайна инициализированы.


In [7]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language='multilingual',
    calculate_probabilities=True,
    verbose=True
)

docs = df_sampled['cleaned_text'].tolist()
print(f"Обучение BERTopic на {len(docs)} документах...")
topics, probs = topic_model.fit_transform(docs)

topic_info = topic_model.get_topic_info()
print(f"\nОбнаружено топиков: {len(topic_info) - 1} (+ Topic -1 — выбросы)")
topic_info.head(15)

2026-04-11 09:05:43,702 - BERTopic - Embedding - Transforming documents to embeddings.


Обучение BERTopic на 30000 документах...


Batches:   0%|          | 0/938 [00:00<?, ?it/s]

2026-04-11 09:07:05,479 - BERTopic - Embedding - Completed ✓
2026-04-11 09:07:05,480 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-11 09:08:01,285 - BERTopic - Dimensionality - Completed ✓
2026-04-11 09:08:01,287 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-11 09:08:13,869 - BERTopic - Cluster - Completed ✓
2026-04-11 09:08:13,885 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-11 09:08:18,519 - BERTopic - Representation - Completed ✓



Обнаружено топиков: 56 (+ Topic -1 — выбросы)


,Topic,Count,Name,Representation,Representative_Docs
0,-1,11591,-1_год_который_человек_россия,"[год, который, человек, россия, также, сообщат...",[власть китай ответить четверг обвинение несоб...
1,0,3463,0_суд_дело_год_уголовный,"[суд, дело, год, уголовный, задержать, убийств...",[владивосток взять стража двое подросток подоз...
2,1,2073,1_матч_команда_сборная_чемпионат,"[матч, команда, сборная, чемпионат, клуб, трен...",[чемпион мир футбол сборная бразилия стать вто...
3,2,1722,2_президент_выборы_украина_россия,"[президент, выборы, украина, россия, партия, п...",[кремль вестись работа подготовка президентски...
4,3,1248,3_самолёт_авиакомпания_аэропорт_полёт,"[самолёт, авиакомпания, аэропорт, полёт, верто...",[разбиться южный судан грузовой самолёт числит...
5,4,810,4_банк_процент_миллиард_год,"[банк, процент, миллиард, год, доллар, кредит,...",[апрель год рост денежный предложение годовой ...
6,5,504,5_фильм_картина_роль_режиссёр,"[фильм, картина, роль, режиссёр, актёр, съёмка...",[актёр билл мюррей решить сняться фильм «охотн...
7,6,461,6_израиль_израильский_палестинский_палестинец,"[израиль, израильский, палестинский, палестине...",[сентябрь около москва контрольный пропускной ...
8,7,459,7_газ_нефть_газпром_украина,"[газ, нефть, газпром, украина, кубометр, цена,...",[стоимость российский газ украина прогноз госк...
9,8,417,8_музей_памятник_картина_художник,"[музей, памятник, картина, художник, искусство...",[полотно прачка написать тулуза лотрек год про...


## 5. Визуализация результатов (2 балла)

### 5.1. Топ-токены для каждого топика

In [8]:
import plotly.io as pio
pio.renderers.default = 'colab'

# Топ-токены по топикам (barchart)
fig = topic_model.visualize_barchart(top_n_topics=10, n_words=10, title='Топ-10 тем: ключевые токены')
fig.show()

### 5.2. Документы с их топиками в 2D пространстве

In [9]:
# Intertopic Distance Map — документы/топики в 2D
fig = topic_model.visualize_topics(title='Карта топиков (Intertopic Distance Map)')
fig.show()

### 5.3. Распределение тем по токенам для выборочных текстов

In [10]:
import numpy as np

sample_indices = [0, 5, 10]

for idx in sample_indices:
    original_text = df_sampled.iloc[idx]['text'][:200] + "..."
    print(f"\n--- Документ {idx} ---")
    print(f"Текст: {original_text}")

    fig = topic_model.visualize_distribution(probs[idx], min_probability=0.01)
    fig.update_layout(title=f"Распределение тем для документа {idx}")
    fig.show()


--- Документ 0 ---
Текст: Египетский перевозчик EgyptAir сообщил о возможном повышении стоимости билетов на свои международные рейсы из‑за девальвации национальной валюты. Такое заявление сделал генеральный директор перевозчик...



--- Документ 5 ---
Текст: Российский вице-премьер Виталий Мутко пожизненно отстранен от участия в Олимпийских играх. Такое решение принял исполком Международного олимпийского комитета (МОК) в связи с допинговым скандалом в Рос...



--- Документ 10 ---
Текст: Египетский суд 13 сентября вынес приговор по делу бывшего премьер-министра страны Ахмеда Назифа, обвинявшегося в "незаконном обогащении", сообщает Agence France-Presse. Отставной чиновник был признан ...


## 6. Оценка качества (2 балла)

### 6.1. Topic Diversity

In [11]:
def calculate_topic_diversity(model, top_n=10):
    all_topics = model.get_topics()
    topic_words = [
        [word for word, _ in all_topics[topic][:top_n]]
        for topic in all_topics if topic != -1
    ]
    if not topic_words:
        return 0.0
    flat_words = [word for sublist in topic_words for word in sublist]
    unique_words = set(flat_words)
    return len(unique_words) / len(flat_words)

diversity = calculate_topic_diversity(topic_model)
print(f"Topic Diversity Score: {diversity:.4f}")
print(f"\n{diversity*100:.1f}% топ-ключевых слов уникальны across всех тем.")
print("Значение ближе к 1.0 означает, что темы хорошо разделены и не пересекаются по лексике.")

Topic Diversity Score: 0.8393

83.9% топ-ключевых слов уникальны across всех тем.
Значение ближе к 1.0 означает, что темы хорошо разделены и не пересекаются по лексике.


### 6.2. UMass Coherence

In [12]:
from gensim import corpora
from gensim.models.coherencemodel import CoherenceModel

# Токенизация
tokenized_docs = [doc.split() for doc in df_sampled['cleaned_text'].tolist()]

# Словарь и корпус Gensim
dictionary = corpora.Dictionary(tokenized_docs)
dictionary.filter_extremes(no_below=10, no_above=0.5)
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

# Топ-10 слов каждого топика
all_topics = topic_model.get_topics()
topic_words = [
    [word for word, _ in all_topics[topic][:10]]
    for topic in all_topics if topic != -1
]

# UMass Coherence
coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    corpus=corpus,
    coherence='u_mass'
)

umass_score = coherence_model.get_coherence()
print(f"UMass Coherence Score: {umass_score:.4f}")
print(f"\nUMass обычно отрицателен; значения ближе к 0 (например, -1...-2) — хорошая когерентность.")
print("Очень низкие значения (< -10) указывают на плохую связность слов внутри тем.")

UMass Coherence Score: -2.0603

UMass обычно отрицателен; значения ближе к 0 (например, -1...-2) — хорошая когерентность.
Очень низкие значения (< -10) указывают на плохую связность слов внутри тем.


## 7. Анализ результатов и выводы (1 балл)

### Интерпретация метрик

- **Topic Diversity** — высокое значение (ожидаемо ~0.9) говорит о том, что модель выделяет темы с минимальным пересечением ключевых слов. Каждая тема описывается уникальным набором токенов.
- **UMass Coherence** — значение порядка -1...-2 свидетельствует о хорошей семантической когерентности: слова внутри каждой темы действительно часто встречаются вместе в корпусе.

### Роль Topic -1 (выбросы)

HDBSCAN — плотностной алгоритм кластеризации, который не присваивает тему документам из зон низкой плотности, помечая их как Topic -1 (выбросы). Это преимущество перед K-Means: модель явно сообщает, какие документы не вписываются ни в одну тему, вместо того чтобы «впихивать» их в ближайший кластер.

### Что удалось
- Пайплайн BERTopic с мультиязычным энкодером успешно выделяет интерпретируемые темы в русскоязычном корпусе новостей.
- Лемматизация через pymorphy3 существенно улучшает качество c-TF-IDF представлений.
- Визуализации наглядно демонстрируют тематическую структуру данных.

### Проблемы и пути решения
- **Большое количество выбросов (Topic -1):** можно уменьшить `min_cluster_size` или применить стратегию reduce outliers в BERTopic.
- **Масштабируемость:** лемматизация 100k+ текстов занимает значительное время. Решение — использовать `joblib`/`multiprocessing` или обрабатывать батчами.
- **Качество эмбеддингов:** для русского языка можно попробовать более специализированные модели (например, `cointegrated/rubert-tiny2` или `ai-forever/sbert_large_nlu_ru`), что может улучшить кластеризацию.
- **Гиперпараметры:** можно провести более систематический подбор через grid search по `min_cluster_size`, `n_neighbors` и т.д., оценивая метрики когерентности.